In [1]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function
from kdv import *

# Model Initialization Testing

In [2]:
# See if model can be initialized
INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 42, 
    verbose                  = True,
)
model = KDV(INIT_PARAMS)

Using device: cuda


# Loss Function Testing

First step is testing if these functions both run and output the expected results

In [3]:
from kdv_loss import *
from kdv_trainer import *

In [ ]:
weights = {
    'w_ic': 10.7,
    'w_bc': 2.0
    #'w_pde': 20.0 #missing on purpose to test update works correctly
}

domain = setup_training_domain(1000, 100, 100, model.soliton_params)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Test loss dict
losses = init_loss_list()
print(f'The loss dict: {losses}')

#Test loss weights
loss_weights = init_loss_weights(device, init_weights=None)
print(f'The loss wieghts as default: {loss_weights}')
loss_weights = init_loss_weights(device, weights)
print(f'The loss wieghts with dict: {loss_weights}')
print(f'Loss weights device: {loss_weights.device} \n')

#loss_comps = torch.tensor([0.3, 4.3, 1.0], device=loss_weights.device) #dummy var, check loss_components function with proper neural network
loss_comps = loss_components(model.neural_net, domain=domain)
total_loss = torch.dot(loss_weights, loss_comps)
#make sure that the dot operator works as intended
print(f'Total loss: {total_loss}')

The loss dict: {'total': [], 'initial': [], 'boundary': [], 'pde': []}
The loss wieghts as default: tensor([1., 1., 1.], device='cuda:0')
The loss wieghts with dict: tensor([10.7000,  2.0000,  1.0000], device='cuda:0')
Loss weights device: cuda:0 



/home/jairdan/miniconda3/envs/soliton-pinns/lib/python3.14/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


NameError: name 'compute_total_loss' is not defined

# Train Function Testing

In [ ]:
#See if model can be trained
TRAIN_PARAMS = dict(
    adam_epochs              = 1000,
    lbfgs_epochs             = 1200,
    verbose_step             = 100,
    n_collocation            = 50000, 
    n_initial                = 30000,  
    n_boundary               = 10000,    
    adam_lr                  = 1e-3,   
    lbfgs_lr                 = 1.0,    
    lbfgs_history_size       = 100,  
    lbfgs_version            = 'old',
    adaptive_sampling        = False,
    logging                  = True
)

TRAIN_WEIGHTS = dict(
    w_ic                     = 5.0,    
    w_bc                     = 1.0,    
    w_pde                    = 15.0,
)

training_stats = model.train(TRAIN_PARAMS, TRAIN_WEIGHTS)

Starting Adam optimization...
[gpu mem] train start               alloc    31.7 MB  reserved   112.0 MB  peak    31.7 MB


/home/jairdan/soliton-pinns/models_refactor/kdv/kdv_loss.py:110: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses['total'].append(float(total_loss))


Adam - Epoch 0/1000, Total Loss: 2.249786e+00
Adam - Epoch 100/1000, Total Loss: 1.020504e-02
Adam - Epoch 200/1000, Total Loss: 5.162701e-03
Adam - Epoch 300/1000, Total Loss: 2.516173e-03
Adam - Epoch 400/1000, Total Loss: 1.319977e-03
Adam - Epoch 500/1000, Total Loss: 7.871253e-04
Adam - Epoch 600/1000, Total Loss: 5.268813e-04
Adam - Epoch 700/1000, Total Loss: 3.852219e-04
Adam - Epoch 800/1000, Total Loss: 3.016567e-04
Adam - Epoch 900/1000, Total Loss: 2.481635e-04
Adam - Epoch 999/1000, Total Loss: 2.115385e-04
[gpu mem] after Adam                alloc    32.4 MB  reserved   970.0 MB  peak   754.9 MB

Starting L-BFGS optimization...
L-BFGS - Iteration 1/1200, Total Loss: 2.105513e-04
L-BFGS - Iteration 101/1200, Total Loss: 4.826894e-05
L-BFGS - Iteration 201/1200, Total Loss: 2.209556e-05
L-BFGS - Iteration 301/1200, Total Loss: 1.142386e-05
L-BFGS - Iteration 401/1200, Total Loss: 6.689329e-06
L-BFGS - Iteration 501/1200, Total Loss: 3.781863e-06
L-BFGS - Iteration 601/1200,

# Testing Function Testing

In [ ]:
model.test(1000, 1000, 'absolute-normalized')

absolute-normalized error metrics:
Mean: 4.056682e-03
Maximum: 7.911295e-02


True

# Function Profiling

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function
from kdv import *

TRAIN_PARAMS = dict(
    adam_epochs              = 1000,
    verbose_step             = 100,
    n_collocation            = 50000, 
    n_initial                = 30000,  
    n_boundary               = 10000,    
    adam_lr                  = 1e-3,   
    lbfgs_lr                 = 1.0,    
    lbfgs_history_size       = 100,  
    lbfgs_version            = 'old',
    adaptive_sampling        = False,   
)

TRAIN_WEIGHTS = dict(
    w_ic                     = 5.0,    
    w_bc                     = 1.0,    
    w_pde                    = 15.0,
)

INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 42, 
    verbose                  = True,
)

In [ ]:
model = KDV(INIT_PARAMS)

def run():
    #add the sequence of loss function calls you want to run here
    return

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
    record_shapes=True, 
    profile_memory=True,
    with_modules=True,
    acc_events=True
) as p:
    run()

Using device: cuda
